# CSE258R Assignment 2 | Food.com Cold-Start Recipe Rating Prediction

**Research question:** For users with rating history, can we predict the star rating they would give to a recipe with no prior ratings, using ingredients and personal taste?

This notebook summarizes the team pipeline:

| Section | Directory | Content |
| --- | --- | --- |
| 1. Data preparation | `wyt/` | Cleaning, chronological splits, cold-start protocol |
| 2. Baseline models | `baseline/` | Global/user statistics + ingredient TF-IDF Ridge |
| 3. Personalization | `personalization/` | Taste similarity + residual boosting (MSE submit) |

**Primary metric: MSE** (lower is better). Validation uses only rows with `is_cold_start_eval=1`. After locking hyperparameters, models are refit on early+middle data and scored on late first-rating-day samples.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.precision", 4)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)


def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "data_prepared" / "shared").is_dir():
            return p
    raise FileNotFoundError("data_prepared/shared not found: unzip shared data at repo root")


ROOT = find_root()
SHARED = ROOT / "data_prepared" / "shared"
WYT = ROOT / "data_prepared" / "wyt"
BASE_OUT = ROOT / "baseline" / "outputs"
PERS_OUT = ROOT / "personalization" / "outputs"
print("ROOT =", ROOT)

---
## 1. Data preparation

Source: [Food.com Recipes and User Interactions](https://www.kaggle.com/shuyangli94/food-com-recipes-and-user-interactions).  
Reproduction script: `wyt/prepare_foodcom.py`.

### 1.1 Cleaning

1. **Recipes:** keep valid submission dates and non-empty ingredient lists; lowercase/normalize whitespace; store ingredients as a JSON list string.
2. **Interactions:** keep ratings in {1,...,5}; drop ratings before recipe publication; drop 0-star rows.
3. Assign a stable `row_id` from the original interaction CSV (1-indexed).
4. Join ingredients and `submitted` via `id` ↔ `recipe_id`.

After cleaning: **1,070,294** valid ratings (from 1,132,367 raw interactions).

In [ ]:
summary = json.loads((ROOT / "data_prepared" / "data_summary.json").read_text(encoding="utf-8"))
cleaning = pd.Series(summary["interaction_cleaning"], name="count").to_frame()
cleaning["share"] = cleaning["count"] / cleaning.loc["raw", "count"]
print("validation_start =", summary["validation_start"])
print("test_start       =", summary["test_start"])
cleaning

### 1.2 Chronological split (leakage control)

| Stage | Dates | Role |
| --- | --- | --- |
| Train (early) | `< 2010-04-25` | Learn patterns; build user history |
| Middle | `2010-04-25` … `2012-09-10` | Validation / tuning; all rows may be used in final refit |
| Late | `≥ 2012-09-11` | Final test |

**Cold-start eval rows:** the user has history before the cutoff, and the rating date equals the recipe's **first valid rating day** in the data.  
Validation marks these with `is_cold_start_eval`; only rows with value 1 enter validation MAE/MSE.

In [ ]:
rows = []
for name, key in [
    ("train", "train"),
    ("validation_all (middle)", "validation_all"),
    ("validation cold-start", "validation_first_day"),
    ("test_all (late)", "test_all"),
    ("test cold-start", "test_first_day"),
]:
    s = summary["splits"][key]
    dist = s["rating_distribution"]
    rows.append({
        "split": name,
        "ratings": s["ratings"],
        "users": s["users"],
        "recipes": s["recipes"],
        "first_date": s["first_date"],
        "last_date": s["last_date"],
        "share_5star": dist.get("5", 0) / s["ratings"],
    })
split_table = pd.DataFrame(rows).set_index("split")
split_table

In [ ]:
train_head = pd.read_csv(SHARED / "train.csv", nrows=3, dtype={"user_id": str, "recipe_id": str})
val_head = pd.read_csv(SHARED / "validation.csv", nrows=3, dtype={"user_id": str, "recipe_id": str})
test_head = pd.read_csv(SHARED / "test_features.csv", nrows=3, dtype={"user_id": str, "recipe_id": str})
print("train columns:", list(train_head.columns))
print("validation columns:", list(val_head.columns))
print("test_features columns:", list(test_head.columns))
train_head

### 1.3 Evaluation protocol

1. **Validation:** fit on `train.csv` only; score the 24,252 rows with `is_cold_start_eval=1`.
2. **Test:** refit on `train + all validation` (909,740 rows) with locked hyperparameters; predict 11,668 rows in `test_features.csv`.
3. Submission format: `row_id,prediction`. Align with `wyt/test_labels.csv` and compute
   \(\mathrm{MSE}=\mathrm{mean}((\hat y-y)^2)\); report 95% CIs via user-level bootstrap when needed.
4. For each training row, user features may use **only ratings strictly before that row's date**.

---
## 2. Baseline models

Code/docs: `baseline/baseline.ipynb`, `baseline/baseline.md`.  
Metrics below are loaded from exported outputs.

### 2.1 Six baselines

| Model | Method | Tuning |
| --- | --- | --- |
| `global_mean` | Training-set mean rating | — |
| `global_median` | Training-set median (5 in this data) | — |
| `user_mean` | \((\sum r + k\mu)/(|H|+k)\) | \(k=5\) by validation MSE |
| `user_median` | Add \(k\) pseudo counts of the global median, then median | \(k=2\) by validation MAE |
| `ridge_tfidf` | Ingredient TF-IDF (word, min_df=5) + Ridge | `alpha=300` by validation MSE |
| `ridge_tfidf_shifted` | Add a constant shift, clip to [1, 5] | shift +0.65 by validation MAE (collapses to all-5) |

In [ ]:
base_val = pd.read_csv(BASE_OUT / "metrics" / "val_metrics.csv")
base_test = pd.read_csv(BASE_OUT / "metrics" / "test_metrics.csv")
base_period = pd.read_csv(BASE_OUT / "metrics" / "test_metrics_by_period.csv")
base_params = json.loads((BASE_OUT / "tuning" / "params.json").read_text(encoding="utf-8"))
print("baseline tuned params:")
print(json.dumps(base_params, indent=2))

In [ ]:
def show_metrics(df, title):
    cols = [c for c in ["model", "tuned_on", "MAE", "MAE_95CI", "MSE", "MSE_95CI"] if c in df.columns]
    out = df[cols].copy()
    print(title)
    return out

show_metrics(base_val, "Validation (cold-start 24,252)")

In [ ]:
show_metrics(base_test, "Test (cold-start 11,668)")

In [ ]:
mse_wide = base_period.pivot(index="model", columns="period", values="MSE")
mse_wide = mse_wide.reindex(columns=[
    "overall",
    "2012-09-11..2013-09-10",
    "2013-09-11..2018-12-17",
])
mse_wide

### 2.2 Baseline takeaways (MSE)

1. **`user_mean` is the strongest non-content baseline** (test MSE ≈ 0.354): user rating level dominates.
2. **Ingredient-only `ridge_tfidf` barely beats `global_mean`** (0.396 vs 0.402): content signal is weak.
3. About 76% of ratings are 5 stars, so MAE and MSE rank models differently; personalization focuses on **MSE**.
4. Later test periods have higher MSE — frozen user profiles drift over time.

In [ ]:
STARS = np.arange(1, 6, dtype=float)

def predict_user_mean_from_counts(counts: pd.DataFrame, users, global_mean: float, k: float):
    # Shrunk user mean; also used as Model C's hard base
    c = counts.to_numpy(float)
    est = (c @ STARS + k * global_mean) / (c.sum(axis=1) + k)
    return pd.Series(est, index=counts.index).reindex(users).fillna(global_mean).to_numpy()

print("user_mean: (sum ratings + k * global_mean) / (n + k), k=5")

---
## 3. Personalization models

Code: `personalization/features.py`, `models.py`, `run_experiment.py`, `run_model_c.py`.  
Notes: `personalization/personalization.md`.

### 3.1 Shared leakage-safe features

For each sample \((u,i,t)\):

| Feature | Definition |
| --- | --- |
| `user_mean` | Shrunk mean from history with date \(< t\) (\(k=5\)) |
| `sim` | Cosine similarity between taste vector \(p_u=\sum_{j<t}(r_j-\hat\mu_u)x_j\) and recipe vector \(x_i\), shrunk by \(n/(n+\mathrm{sim\_k})\) |
| `log_n` / `n_hist` | History length |
| `global_ridge` | Global ingredient TF-IDF → Ridge score for recipe \(i\) |

Recipe vectors: ingredient TF-IDF (word) → `TruncatedSVD(128)`.  
At validation/test time, user profiles are **frozen** at the fit cutoff.

### 3.2 Model families

- **A:** linear absolute-rating model  
  `Ridge(user_mean, sim, log_n, global_ridge) → rating`
- **B:** median/mean base + linear residual (MAE-oriented; not selected for MSE)
- **C (MSE submit):** hard `user_mean` base; fit residual `r - user_mean` with  
  `HistGradientBoostingRegressor(loss='squared_error')`,  
  plus interactions `sim*log_n`, `global_ridge - user_mean`, `|sim|`

In [ ]:
pers_params = json.loads((PERS_OUT / "tuning" / "params.json").read_text(encoding="utf-8"))
print("submit_model =", pers_params.get("submit_model"), "| family =", pers_params.get("submit_family"))
print("primary_metric =", pers_params.get("primary_metric"))
print("\nbest_mse (Model A):")
print(json.dumps(pers_params["best_mse"], indent=2))
print("\nbest_c (Model C, submitted):")
print(json.dumps(pers_params["best_c"], indent=2))

In [ ]:
pers_val = pd.read_csv(PERS_OUT / "metrics" / "val_metrics_with_c.csv")
pers_test = pd.read_csv(PERS_OUT / "metrics" / "test_metrics_with_c.csv")
pers_period = pd.read_csv(PERS_OUT / "metrics" / "test_metrics_by_period_with_c.csv")
show_metrics(pers_val, "Personalization validation")

In [ ]:
show_metrics(pers_test, "Personalization test")

In [ ]:
pers_period.pivot(index="model", columns="period", values="MSE")

### 3.3 Submitted Model C

1. Compute \(\hat\mu_u=\mathrm{user\_mean}\) under the same protocol as the baseline.
2. Train on residual \(y^{\mathrm{res}}=r-\hat\mu_u\).
3. Fit a boosting tree on residual features (nonlinear interactions).
4. Predict \(\hat r=\mathrm{clip}(\hat\mu_u+\hat\delta,1,5)\).

Locked hyperparameters: `sim_k=20`, `max_depth=4`, `learning_rate=0.05`, `max_iter=200`, `min_samples_leaf=50`.

In [ ]:
submit_pred = pd.read_csv(PERS_OUT / "predictions" / "test_personalization_best_mse.csv")
print("rows =", len(submit_pred), "| columns =", list(submit_pred.columns))
print("prediction mean/std =", submit_pred["prediction"].mean(), submit_pred["prediction"].std())
submit_pred.head()

---
## 4. Overall comparison (MSE)

Baseline and personalization models under the same test protocol.

In [ ]:
board = []
for _, r in base_test.iterrows():
    board.append({"model": r["model"], "group": "baseline", "test_MSE": r["MSE"], "test_MAE": r["MAE"]})
for _, r in pers_test.iterrows():
    board.append({
        "model": r["model"],
        "group": "personalization" if r["model"] != "ref_user_mean" else "baseline-ref",
        "test_MSE": r["MSE"],
        "test_MAE": r["MAE"],
    })

board_df = pd.DataFrame(board)
board_df = board_df[board_df["model"] != "ref_user_mean"].drop_duplicates("model")
board_df = board_df.sort_values("test_MSE").reset_index(drop=True)
um = float(board_df.loc[board_df["model"] == "user_mean", "test_MSE"].iloc[0])
board_df["delta_vs_user_mean"] = board_df["test_MSE"] - um
board_df

In [ ]:
um = float(base_test.loc[base_test["model"] == "user_mean", "MSE"].iloc[0])
pc = float(pers_test.loc[pers_test["model"] == "personalization_C", "MSE"].iloc[0])
pa = float(pers_test.loc[pers_test["model"] == "personalization_A", "MSE"].iloc[0])
print(f"user_mean          test MSE = {um:.4f}")
print(f"personalization A  test MSE = {pa:.4f}   (delta {pa-um:+.4f})")
print(f"personalization C  test MSE = {pc:.4f}   (delta {pc-um:+.4f})  <- submit")

## 5. Conclusions

1. **Data:** chronological splits + first-rating-day cold start prevent future leakage into user profiles or recipe popularity.
2. **Baseline:** under MSE, `user_mean` wins; ingredient-only Ridge adds little.
3. **Personalization:** adding taste similarity and a global ingredient score on top of `user_mean`, then fitting residual boosting (**Model C**), reaches test MSE ≈ **0.3491**, better than `user_mean` (0.3536) and linear Model A (0.3495).
4. **Limits:** ratings are heavily 5-star; content signal is weak; frozen profiles degrade in later periods.

### Key paths

| Purpose | Path |
| --- | --- |
| Data prep | `wyt/prepare_foodcom.py` |
| Baseline | `baseline/baseline.ipynb` |
| Personalization A/B | `personalization/run_experiment.py` |
| Personalization C | `personalization/run_model_c.py` |
| Submitted predictions | `personalization/outputs/predictions/test_personalization_best_mse.csv` |